In [35]:
import torch
import gc
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

test
CUDA available: True
GPU: Tesla T4
dissertation.ipynb


In [ ]:
!pip install transformers accelerate pillow safetensors controlnet-aux diffusers


In [ ]:
from PIL import Image
import numpy as np
import cv2

In [ ]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from google.colab import files
from IPython.display import display
from PIL import Image
import io

print("Please upload an image")
uploaded = files.upload()

image_name = list(uploaded.keys())[0]
image = Image.open(io.BytesIO(uploaded[image_name])).convert("RGB")

print("Loading small BLIP-2 model (opt-2.7b)...")

display(image)
print(f"Loaded image: {image_name}")


#BLIP2
print("\nLoading BLIP-2 model")
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16,
    device_map="auto",
)

inputs = processor(image, return_tensors="pt").to(model.device)
ids = model.generate(**inputs, max_length=40)
caption = processor.decode(ids[0], skip_special_tokens=True)

print("\nCaption:", caption)

del model
del processor
gc.collect()
torch.cuda.empty_cache()

In [ ]:
#MiDaS
midas_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


midas_model = torch.hub.load("intel-isl/MiDaS", "DPT_Hybrid")
midas_model.to(midas_device).eval()

midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = midas_transforms.dpt_transform

img_np = np.array(image)
input_batch = transform(img_np).to(midas_device)

with torch.no_grad():
    prediction = midas_model(input_batch)
    prediction = torch.nn.functional.interpolate(
        prediction.unsqueeze(1),
        size=img_np.shape[:2],
        mode="bicubic",
        align_corners=False
    ).squeeze(0).squeeze(0)

depth = prediction.detach().cpu().numpy()

# Normalize to 0..255 for saving/viewing
depth_vis = cv2.normalize(depth, None, 0, 255, norm_type=cv2.NORM_MINMAX)
depth_vis = depth_vis.astype(np.uint8)

depth_color = cv2.applyColorMap(depth_vis, cv2.COLORMAP_INFERNO)

out_path_gray = "depth_gray.png"
out_path_color = "depth_color.png"
cv2.imwrite(out_path_gray, depth_vis)
cv2.imwrite(out_path_color, depth_color)

print("\nDepth saved to:", out_path_gray, out_path_color)
display(Image.open(out_path_gray))
display(Image.open(out_path_color))

del midas_model
torch.cuda.empty_cache()

In [ ]:
from diffusers import ControlNetModel, StableDiffusionControlNetImg2ImgPipeline

gc.collect()
torch.cuda.empty_cache()

init_image = image.copy().convert("RGB")
depth_image = Image.open("depth_gray.png").convert("RGB")
#depth_image = Image.open("depth_color.png").convert("RGB")

print("Upload conditional image for IP-Adapter")
uploaded_conditional = files.upload()

cond_name = list(uploaded_conditional.keys())[0]
ip_image = Image.open(io.BytesIO(uploaded_conditional[cond_name])).convert("RGB")
display(ip_image)
print("Loaded conditional image:", cond_name)

#resize image for consistency
init_image = init_image.resize((512,512))
depth_image = depth_image.resize((512,512))
ip_image= ip_image.resize((512,512))

#user inputs and blip2 output caption
user_prompt = input("Enter the transformation you want (e.g. 'snowy winter scene', 'cartoon style', 'oil painting'): ")

final_prompt = f"Scene description: {caption}. User prompt: {user_prompt}."
negative_prompt = "blurry, low quality, warped perspective, distorted geometry, extra limbs, deformed face, bad anatomy, warped perspective, artifacts"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("Loading controlNet")
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth",
    torch_dtype=dtype
)

#load model
print("Loading stable diffusion")
pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None
).to(device)


#pipe.enable_attention_slicing()

#load IP adapter
print("Loading IP adapter")
pipe.load_ip_adapter(
    "h94/IP-Adapter",
    subfolder="models",
    weight_name="ip-adapter_sd15.bin"
)

#balance text and image prompt
pipe.set_ip_adapter_scale(0.3)

try:
    pipe.enable_xformers_memory_efficient_attention()
    print("xFormers enabled")
except Exception as e:
    print("xFormers not available:", e)

generator = torch.Generator(device=device).manual_seed(42)


print("Generating final image")
out = pipe(
    prompt=final_prompt,
    negative_prompt=negative_prompt,
    image=init_image,
    control_image=depth_image,
    ip_adapter_image=ip_image,
    strength=0.85,
    guidance_scale=8.5,
    controlnet_conditioning_scale=0.8,
    num_inference_steps=40,
    generator=generator
).images[0]

out.save("translated.png")
display(out)



In [26]:
print("Device:", pipe.device)
print("UNet dtype:", pipe.unet.dtype)

NameError: name 'pipe' is not defined

In [48]:
from getpass import getpass
token = getpass("Paste new GitHub token: ")

!git remote set-url origin https://{token}@github.com/josephzl04/image-data-generation.git
!git push origin main
!git remote set-url origin https://github.com/josephzl04/image-data-generation.git

Paste new GitHub token: ··········
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (8/8), 4.33 KiB | 4.33 MiB/s, done.
Total 8 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 1 local object.
To https://github.com/josephzl04/image-data-generation.git
   aad88be..e9221e2  main -> main
